# Interactive Power Notebook: Single‑Phase and Three‑Phase (Pedagogical)

This notebook is an **interactive visualization tool** for understanding:

- **Phasors** (RMS magnitude + angle)
- **Complex power**: \(S = P + jQ\)
- **Power factor** (leading/lagging)
- **Time-domain waveforms**: \(v(t)\), \(i(t)\), instantaneous power \(p(t)\)
- **Averaging / integration windows** for \(p(t)\) and energy

It includes two interactive sections:

1. **Single-phase**
2. **Three-phase (balanced sinusoidal)**

---

## Assumptions and conventions

### Phasors (RMS)
We treat the sliders \(|V|\), \(\angle V\), \(|I|\), \(\angle I\) as **RMS phasors**:

\[
\underline{V} = |V|\,e^{j\theta_V},\qquad \underline{I} = |I|\,e^{j\theta_I}
\]

### Complex power
**Single-phase complex power** (RMS convention):

\[
\underline{S} = \underline{V}\,\underline{I}^* = P + jQ
\]

So:

\[
P = |V||I|\cos(\theta_V - \theta_I),\qquad
Q = |V||I|\sin(\theta_V - \theta_I)
\]

**Apparent power** magnitude:

\[
|S| = |V||I|
\]

**Power factor**:

\[
\mathrm{pf} = \frac{P}{|S|} = \cos(\theta_V-\theta_I)
\]

### Time domain waveforms
Given RMS phasors and frequency \(f\) with \(\omega = 2\pi f\), we generate:

\[
v(t) = \sqrt{2}|V|\cos(\omega t + \theta_V),\qquad
i(t) = \sqrt{2}|I|\cos(\omega t + \theta_I)
\]

Instantaneous power:

\[
p(t) = v(t)\,i(t)
\]

### Averaging / integration window
You can pick a **window start** and **window length** (in cycles). The notebook shows:

- Average power over the selected window:
\[
\overline{p} = \frac{1}{T_w}\int_{t_0}^{t_0+T_w} p(t)\,dt
\]
- Energy over the same window:
\[
E = \int_{t_0}^{t_0+T_w} p(t)\,dt
\]

---

## If widgets don’t render
In classic Jupyter:
- `pip install ipywidgets`
- Enable the extension as needed for your environment (JupyterLab vs Notebook).



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout, Tab, interactive_output

# ---------- Helpers ----------
def deg2rad(deg):
    return np.deg2rad(deg)

def phasor(mag_rms, ang_deg):
    return mag_rms * np.exp(1j * deg2rad(ang_deg))

def complex_power(V_rms, I_rms):
    # S = V * conj(I)
    return V_rms * np.conj(I_rms)

def power_factor(thetaV_deg, thetaI_deg):
    phi = deg2rad(thetaV_deg - thetaI_deg)
    return np.cos(phi)

def lead_lag(thetaV_deg, thetaI_deg, tol_deg=1e-9):
    d = (thetaV_deg - thetaI_deg)
    # Wrap to (-180, 180]
    d = (d + 180) % 360 - 180
    if abs(d) < tol_deg:
        return "in-phase"
    return "lagging (inductive)" if d > 0 else "leading (capacitive)"

def make_timebase(f_hz, cycles=3.0, ppc=1200):
    # cycles: number of fundamental cycles to show
    T = 1.0 / f_hz
    N = int(np.ceil(cycles * ppc))
    t = np.linspace(0.0, cycles * T, N, endpoint=False)
    return t

def window_indices(t, f_hz, start_cycles, window_cycles):
    T = 1.0 / f_hz
    t0 = start_cycles * T
    tw = window_cycles * T
    t1 = t0 + tw
    i0 = np.searchsorted(t, t0, side="left")
    i1 = np.searchsorted(t, t1, side="left")
    i0 = np.clip(i0, 0, len(t)-1)
    i1 = np.clip(i1, i0+1, len(t))
    return i0, i1, t0, t1

def trapz_avg(x, t, i0, i1):
    # Average over [i0, i1)
    area = np.trapz(x[i0:i1], t[i0:i1])
    dt = t[i1-1] - t[i0] if (i1 - i0) > 1 else (t[1]-t[0])
    duration = t[i1-1] - t[i0] + (t[1]-t[0])  # approx includes last sample width
    avg = area / duration
    return avg, area, duration


## 1) Single‑Phase Interactive Explorer

Use the sliders to set **RMS phasors** \(\underline{V}\) and \(\underline{I}\).  
The figures update in real time:

- **Phasor diagram** (V and I)
- **Power triangle** (P, Q, S)
- **Time-domain** \(v(t)\), \(i(t)\)
- **Instantaneous power** \(p(t)\) with a configurable averaging window



In [ ]:
# ---------- Single-phase widgets ----------
w_Vmag = widgets.FloatSlider(value=120.0, min=0.0, max=480.0, step=1.0, description='|V| RMS', continuous_update=False)
w_Vang = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='∠V (deg)', continuous_update=False)

w_Imag = widgets.FloatSlider(value=10.0, min=0.0, max=200.0, step=0.5, description='|I| RMS', continuous_update=False)
w_Iang = widgets.FloatSlider(value=-30.0, min=-180.0, max=180.0, step=1.0, description='∠I (deg)', continuous_update=False)

w_f = widgets.FloatSlider(value=60.0, min=1.0, max=400.0, step=1.0, description='f (Hz)', continuous_update=False)

w_cycles = widgets.FloatSlider(value=3.0, min=0.5, max=8.0, step=0.5, description='show cycles', continuous_update=False)
w_ppc = widgets.IntSlider(value=1200, min=100, max=4000, step=100, description='pts/cycle', continuous_update=False)

w_start = widgets.FloatSlider(value=0.0, min=0.0, max=7.5, step=0.05, description='win start (cyc)', continuous_update=False)
w_wincyc = widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1, description='win len (cyc)', continuous_update=False)

# ---------- Single-phase plot function ----------
def render_single_phase(Vmag, Vang, Imag, Iang, f_hz, cycles, ppc, start_cycles, win_cycles):
    V = phasor(Vmag, Vang)
    I = phasor(Imag, Iang)
    S = complex_power(V, I)
    P = np.real(S)
    Q = np.imag(S)
    pf = P / (np.abs(S) + 1e-15)
    ll = lead_lag(Vang, Iang)

    # time domain
    t = make_timebase(f_hz, cycles=cycles, ppc=ppc)
    w = 2*np.pi*f_hz
    v = np.sqrt(2)*Vmag*np.cos(w*t + deg2rad(Vang))
    i = np.sqrt(2)*Imag*np.cos(w*t + deg2rad(Iang))
    p = v*i

    # window
    # keep window start within range
    start_cycles = min(max(start_cycles, 0.0), max(0.0, cycles - 0.1))
    win_cycles = min(max(win_cycles, 0.1), cycles - start_cycles + 1e-9)
    i0, i1, t0, t1 = window_indices(t, f_hz, start_cycles, win_cycles)
    p_avg, energy, duration = trapz_avg(p, t, i0, i1)

    # Figures
    plt.close('all')
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1])

    # (1) Phasor diagram for V and I
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.axhline(0, linewidth=0.8)
    ax1.axvline(0, linewidth=0.8)
    ax1.set_title("Phasor diagram (RMS): V and I")
    ax1.set_xlabel("Real")
    ax1.set_ylabel("Imag")

    # Scale so both fit nicely
    rmax = max(np.abs(V), np.abs(I), 1e-9) * 1.2
    ax1.set_xlim(-rmax, rmax)
    ax1.set_ylim(-rmax, rmax)
    ax1.set_aspect('equal', adjustable='box')

    ax1.quiver(0, 0, np.real(V), np.imag(V), angles='xy', scale_units='xy', scale=1, width=0.008, label='V')
    ax1.quiver(0, 0, np.real(I), np.imag(I), angles='xy', scale_units='xy', scale=1, width=0.008, label='I')
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)

    # (2) Power triangle
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.axhline(0, linewidth=0.8)
    ax2.axvline(0, linewidth=0.8)
    ax2.set_title("Power triangle: S = P + jQ")
    ax2.set_xlabel("P (W)")
    ax2.set_ylabel("Q (var)")
    ax2.set_aspect('equal', adjustable='box')

    # Draw P vector and Q component
    ax2.quiver(0, 0, P, 0, angles='xy', scale_units='xy', scale=1, width=0.008)
    ax2.quiver(P, 0, 0, Q, angles='xy', scale_units='xy', scale=1, width=0.008)
    ax2.quiver(0, 0, P, Q, angles='xy', scale_units='xy', scale=1, width=0.010, label='S')

    smax = max(abs(P), abs(Q), abs(S), 1e-9) * 1.2
    ax2.set_xlim(-smax, smax)
    ax2.set_ylim(-smax, smax)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper left')

    # Text box with key numbers
    txt = (
        f"V = {Vmag:.3g} ∠ {Vang:.1f}° (RMS)\n"
        f"I = {Imag:.3g} ∠ {Iang:.1f}° (RMS)\n"
        f"S = {P:.3g} + j{Q:.3g}  VA\n"
        f"|S| = {abs(S):.3g} VA\n"
        f"pf = {pf:.4f}  ({ll})\n"
        f"Avg over window: P̄ = {p_avg:.3g} W\n"
        f"Energy over window: E = {energy:.3g} J"
    )
    ax2.text(0.02, 0.02, txt, transform=ax2.transAxes, va='bottom', ha='left',
             bbox=dict(boxstyle='round', alpha=0.15))

    # (3) v(t) and i(t)
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_title("Time domain: v(t) and i(t)")
    ax3.plot(t, v, label='v(t)')
    ax3.plot(t, i, label='i(t)')
    ax3.set_xlabel("t (s)")
    ax3.set_ylabel("Amplitude")
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='upper right')

    # (4) p(t) and averaging window
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_title("Instantaneous power: p(t) = v(t)i(t)")
    ax4.plot(t, p, label='p(t)')
    ax4.axvspan(t0, t1, alpha=0.15, label='avg window')
    ax4.hlines(p_avg, t[0], t[-1], linewidth=2, label='avg over window')
    ax4.set_xlabel("t (s)")
    ax4.set_ylabel("W")
    ax4.grid(True, alpha=0.3)
    ax4.legend(loc='upper right')

    fig.tight_layout()
    plt.show()

# Layout
controls_left = VBox([w_Vmag, w_Vang, w_Imag, w_Iang], layout=Layout(width='48%'))
controls_right = VBox([w_f, w_cycles, w_ppc, w_start, w_wincyc], layout=Layout(width='48%'))
ui_single = HBox([controls_left, controls_right])

out_single = interactive_output(
    render_single_phase,
    dict(
        Vmag=w_Vmag, Vang=w_Vang, Imag=w_Imag, Iang=w_Iang,
        f_hz=w_f, cycles=w_cycles, ppc=w_ppc,
        start_cycles=w_start, win_cycles=w_wincyc
    )
)

display(ui_single, out_single)


## 2) Three‑Phase Interactive Explorer (Balanced Sinusoids)

This section assumes a **balanced three‑phase set**. You set a single \(|V|\), \(\angle V\), \(|I|\), \(\angle I\) and the notebook constructs:

Positive-sequence (ABC) voltages:
\[
\underline{V}_a = |V|e^{j\theta_V},\quad
\underline{V}_b = |V|e^{j(\theta_V-120^\circ)},\quad
\underline{V}_c = |V|e^{j(\theta_V+120^\circ)}
\]

Balanced currents (same shift on each phase):
\[
\underline{I}_a = |I|e^{j\theta_I},\quad
\underline{I}_b = |I|e^{j(\theta_I-120^\circ)},\quad
\underline{I}_c = |I|e^{j(\theta_I+120^\circ)}
\]

Per-phase and total complex power:

\[
\underline{S}_\phi = \underline{V}_a\underline{I}_a^*,\qquad
\underline{S}_{3\phi} = 3\underline{S}_\phi
\]

Key pedagogical highlight:
- Each phase’s instantaneous \(p_a(t)\) has a **2× frequency ripple**
- In a balanced set, the three ripples **cancel**, making **total instantaneous power** nearly constant.

> Note on voltage meaning: you can choose whether \(|V|\) is interpreted as **line-to-neutral (phase RMS)** or **line-to-line RMS**.  
> If line-to-line is selected, the notebook converts \(V_\phi = V_{LL}/\sqrt{3}\).



In [ ]:
# ---------- Three-phase widgets ----------
w3_Vmag = widgets.FloatSlider(value=208.0, min=0.0, max=1000.0, step=1.0, description='|V| RMS', continuous_update=False)
w3_Vang = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='∠V (deg)', continuous_update=False)

w3_Imag = widgets.FloatSlider(value=10.0, min=0.0, max=400.0, step=0.5, description='|I| RMS', continuous_update=False)
w3_Iang = widgets.FloatSlider(value=-30.0, min=-180.0, max=180.0, step=1.0, description='∠I (deg)', continuous_update=False)

w3_f = widgets.FloatSlider(value=60.0, min=1.0, max=400.0, step=1.0, description='f (Hz)', continuous_update=False)

w3_cycles = widgets.FloatSlider(value=2.0, min=0.5, max=8.0, step=0.5, description='show cycles', continuous_update=False)
w3_ppc = widgets.IntSlider(value=1200, min=100, max=4000, step=100, description='pts/cycle', continuous_update=False)

w3_start = widgets.FloatSlider(value=0.0, min=0.0, max=7.5, step=0.05, description='win start (cyc)', continuous_update=False)
w3_wincyc = widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1, description='win len (cyc)', continuous_update=False)

w3_seq = widgets.Dropdown(options=[("ABC (positive seq)", "ABC"), ("ACB (negative seq)", "ACB")],
                          value="ABC", description="sequence")

w3_vdef = widgets.Dropdown(options=[("Line-to-neutral (phase RMS)", "phase"),
                                    ("Line-to-line RMS (convert)", "line")],
                           value="phase", description="|V| means")

# ---------- Three-phase plot function ----------
def render_three_phase(Vmag_in, Vang, Imag, Iang, f_hz, cycles, ppc, start_cycles, win_cycles, sequence, vdef):
    # Interpret voltage magnitude
    if vdef == "line":
        Vmag = Vmag_in / np.sqrt(3)  # convert to phase RMS
        v_label = f"|V|={Vmag_in:.3g} (LL RMS) → {Vmag:.3g} (phase RMS)"
    else:
        Vmag = Vmag_in
        v_label = f"|V|={Vmag:.3g} (phase RMS)"

    # Sequence sign: ABC uses -120 for b and +120 for c; ACB swaps
    if sequence == "ABC":
        b_shift = -120.0
        c_shift = +120.0
    else:  # ACB
        b_shift = +120.0
        c_shift = -120.0

    Va = phasor(Vmag, Vang)
    Vb = phasor(Vmag, Vang + b_shift)
    Vc = phasor(Vmag, Vang + c_shift)

    Ia = phasor(Imag, Iang)
    Ib = phasor(Imag, Iang + b_shift)
    Ic = phasor(Imag, Iang + c_shift)

    Sphi = complex_power(Va, Ia)
    S3 = 3 * Sphi
    P3 = np.real(S3)
    Q3 = np.imag(S3)
    pf = P3 / (np.abs(S3) + 1e-15)
    ll = lead_lag(Vang, Iang)

    # time domain
    t = make_timebase(f_hz, cycles=cycles, ppc=ppc)
    w = 2*np.pi*f_hz

    va = np.sqrt(2)*Vmag*np.cos(w*t + deg2rad(Vang))
    vb = np.sqrt(2)*Vmag*np.cos(w*t + deg2rad(Vang + b_shift))
    vc = np.sqrt(2)*Vmag*np.cos(w*t + deg2rad(Vang + c_shift))

    ia = np.sqrt(2)*Imag*np.cos(w*t + deg2rad(Iang))
    ib = np.sqrt(2)*Imag*np.cos(w*t + deg2rad(Iang + b_shift))
    ic = np.sqrt(2)*Imag*np.cos(w*t + deg2rad(Iang + c_shift))

    pa = va*ia
    pb = vb*ib
    pc = vc*ic
    p_total = pa + pb + pc

    # window
    start_cycles = min(max(start_cycles, 0.0), max(0.0, cycles - 0.1))
    win_cycles = min(max(win_cycles, 0.1), cycles - start_cycles + 1e-9)
    i0, i1, t0, t1 = window_indices(t, f_hz, start_cycles, win_cycles)
    p_avg, energy, duration = trapz_avg(p_total, t, i0, i1)

    # Figures
    plt.close('all')
    fig = plt.figure(figsize=(13, 9))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.1])

    # (1) Phasor diagram: Va,Vb,Vc and Ia,Ib,Ic
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.axhline(0, linewidth=0.8)
    ax1.axvline(0, linewidth=0.8)
    ax1.set_title("3ϕ phasors (RMS): voltages and currents")
    ax1.set_xlabel("Real")
    ax1.set_ylabel("Imag")
    ax1.set_aspect('equal', adjustable='box')

    rmax = max(np.abs(Va), np.abs(Ia), 1e-9) * 1.4
    ax1.set_xlim(-rmax, rmax)
    ax1.set_ylim(-rmax, rmax)

    # Voltages
    ax1.quiver(0, 0, np.real(Va), np.imag(Va), angles='xy', scale_units='xy', scale=1, width=0.007, label='Va')
    ax1.quiver(0, 0, np.real(Vb), np.imag(Vb), angles='xy', scale_units='xy', scale=1, width=0.007, label='Vb')
    ax1.quiver(0, 0, np.real(Vc), np.imag(Vc), angles='xy', scale_units='xy', scale=1, width=0.007, label='Vc')

    # Currents (slightly thicker)
    ax1.quiver(0, 0, np.real(Ia), np.imag(Ia), angles='xy', scale_units='xy', scale=1, width=0.010, label='Ia')
    ax1.quiver(0, 0, np.real(Ib), np.imag(Ib), angles='xy', scale_units='xy', scale=1, width=0.010, label='Ib')
    ax1.quiver(0, 0, np.real(Ic), np.imag(Ic), angles='xy', scale_units='xy', scale=1, width=0.010, label='Ic')

    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left', ncol=2)

    # (2) Power triangle for total 3ϕ
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.axhline(0, linewidth=0.8)
    ax2.axvline(0, linewidth=0.8)
    ax2.set_title("Total power triangle: S₃ϕ = P₃ϕ + jQ₃ϕ")
    ax2.set_xlabel("P (W)")
    ax2.set_ylabel("Q (var)")
    ax2.set_aspect('equal', adjustable='box')

    ax2.quiver(0, 0, P3, 0, angles='xy', scale_units='xy', scale=1, width=0.008)
    ax2.quiver(P3, 0, 0, Q3, angles='xy', scale_units='xy', scale=1, width=0.008)
    ax2.quiver(0, 0, P3, Q3, angles='xy', scale_units='xy', scale=1, width=0.010, label='S₃ϕ')

    smax = max(abs(P3), abs(Q3), abs(S3), 1e-9) * 1.2
    ax2.set_xlim(-smax, smax)
    ax2.set_ylim(-smax, smax)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper left')

    txt = (
        f"{v_label}\n"
        f"I_phase = {Imag:.3g} ∠ {Iang:.1f}° (RMS)\n"
        f"Sequence: {sequence}\n"
        f"S₃ϕ = {P3:.3g} + j{Q3:.3g}  VA\n"
        f"|S₃ϕ| = {abs(S3):.3g} VA\n"
        f"pf = {pf:.4f}  ({ll})\n"
        f"Avg over window: P̄ = {p_avg:.3g} W\n"
        f"Energy over window: E = {energy:.3g} J"
    )
    ax2.text(0.02, 0.02, txt, transform=ax2.transAxes, va='bottom', ha='left',
             bbox=dict(boxstyle='round', alpha=0.15))

    # (3) Phase voltages (time)
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_title("Phase voltages: v_a(t), v_b(t), v_c(t)")
    ax3.plot(t, va, label='va')
    ax3.plot(t, vb, label='vb')
    ax3.plot(t, vc, label='vc')
    ax3.set_xlabel("t (s)")
    ax3.set_ylabel("V")
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='upper right', ncol=3)

    # (4) Phase currents (time)
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_title("Phase currents: i_a(t), i_b(t), i_c(t)")
    ax4.plot(t, ia, label='ia')
    ax4.plot(t, ib, label='ib')
    ax4.plot(t, ic, label='ic')
    ax4.set_xlabel("t (s)")
    ax4.set_ylabel("A")
    ax4.grid(True, alpha=0.3)
    ax4.legend(loc='upper right', ncol=3)

    # (5) Per-phase instantaneous powers
    ax5 = fig.add_subplot(gs[2, 0])
    ax5.set_title("Per-phase instantaneous powers (note 2× ripple)")
    ax5.plot(t, pa, label='pa')
    ax5.plot(t, pb, label='pb')
    ax5.plot(t, pc, label='pc')
    ax5.set_xlabel("t (s)")
    ax5.set_ylabel("W")
    ax5.grid(True, alpha=0.3)
    ax5.legend(loc='upper right', ncol=3)

    # (6) Total instantaneous power with averaging window
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.set_title("Total instantaneous power: p_total(t) = pa+pb+pc")
    ax6.plot(t, p_total, label='p_total')
    ax6.axvspan(t0, t1, alpha=0.15, label='avg window')
    ax6.hlines(p_avg, t[0], t[-1], linewidth=2, label='avg over window')
    ax6.set_xlabel("t (s)")
    ax6.set_ylabel("W")
    ax6.grid(True, alpha=0.3)
    ax6.legend(loc='upper right')

    fig.tight_layout()
    plt.show()

controls3_left = VBox([w3_Vmag, w3_Vang, w3_Imag, w3_Iang], layout=Layout(width='48%'))
controls3_right = VBox([w3_seq, w3_vdef, w3_f, w3_cycles, w3_ppc, w3_start, w3_wincyc], layout=Layout(width='48%'))
ui_three = HBox([controls3_left, controls3_right])

out_three = interactive_output(
    render_three_phase,
    dict(
        Vmag_in=w3_Vmag, Vang=w3_Vang, Imag=w3_Imag, Iang=w3_Iang,
        f_hz=w3_f, cycles=w3_cycles, ppc=w3_ppc,
        start_cycles=w3_start, win_cycles=w3_wincyc,
        sequence=w3_seq, vdef=w3_vdef
    )
)

display(ui_three, out_three)


## Extra ideas (optional extensions you can add later)

If you want to push this notebook further as a teaching aid, natural extensions include:

- **Unbalanced 3ϕ**: independent magnitudes/angles per phase, and show how \(p_{total}(t)\) gains ripple.
- **Neutral current**: for 4-wire Y systems with unbalance.
- **Line-to-line voltages** and **delta-connected currents** explicitly (instead of just converting \(|V|\)).
- **Symmetrical components**: compute \(V^+, V^-, V^0\) and show how sequence content affects power.
- **dq0 / Park transform** view: show how \(P,Q\) relate to \(d\)- and \(q\)-axis components.
- **Power flow sign conventions**: generator vs load sign, or IEC vs IEEE reactive sign.
- **Harmonics**: add THD, show how non-sinusoidal waveforms affect \(S\), \(P\), \(Q\), and distortion power.

If you tell me which of these you want, I can extend the notebook accordingly.
